<a href="https://colab.research.google.com/github/a-forty-two/vodafonecairo31aug/blob/main/Prerequisite-Quantization-CaseStudy_and_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mastering LLM Deployment
## Lab 1 - Foundations, the Case Study, and a Measured Baseline

**Duration:** ~90 minutes  ·  **Runtime required:** `Runtime → Change runtime type → T4 GPU`

---

### Where this lab sits

Day 1 is about **making the model smaller**. Day 2 is about **getting the smaller model into production on AWS ECS**. Nothing on Day 2 works unless Day 1 produces a real, measured, exportable artifact - so this first lab establishes the two things every later lab depends on:

1. A **baseline model** that is deliberately too expensive to serve.
2. A **measurement discipline** - a shared benchmark harness and an *optimization ledger* - so that every claim we make later ("2.5× faster", "4× smaller") is a number we produced, not a number we hoped for.

### Learning outcomes

By the end of this lab you will be able to:

- Refresh the TensorFlow and Keras constructs that matter for deployment work: `tf.data`, the three Keras model APIs, `GradientTape`, `tf.function`, and the `SavedModel` serving signature.
- Analyse a real production LLM deployment and reconstruct the reasoning behind its optimization decisions.
- Build a cost model that turns latency and throughput into monthly dollars.
- Fine-tune a BERT-base text classifier and benchmark it properly - with warm-up, percentiles, and separate batch-size regimes.
- Record baseline results into a persistent ledger that Labs 2–5 will extend.

### The lab sequence

| Lab | Topic | Produces |
|---|---|---|
| **1 (this one)** | Foundations, case study, baseline | `teacher-bert-sst2`, ledger row `baseline` |
| 2 | Model distillation (SQuAD) | `student-squad-4L`, reusable KD utilities |
| 3 | Model quantization (IMDB) | int8 / fp16 variants, TFLite artifacts |
| 4 | Model pruning (SST-2) | sparse models, sparsity-vs-accuracy curve |
| 5 | Capstone: stack all three | the artifact Day 2 deploys to ECS |

---

### A note on running order

Cells are meant to be run **top to bottom, once**. Where a cell is expensive, its expected wall-clock time on a T4 is stated in the markdown above it. Total GPU time for this notebook is roughly 8–12 minutes.

---
## 0. Environment setup

### 0.1 Confirm you have a GPU

If the next cell prints `No GPU`, stop and switch the runtime type. Fine-tuning BERT-base on CPU in this lab would take well over an hour.

In [ ]:
import subprocess, sys, platform

print("Python:", platform.python_version())
try:
    print(
        subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]
        )
        .decode()
        .strip()
    )
except Exception:
    print("No GPU detected -> Runtime > Change runtime type > T4 GPU")

### 0.2 Install the pinned dependency set

Three constraints drive these pins, and they are worth understanding because they will bite you in your own projects:

- **`transformers < 5`** - TensorFlow model classes (`TFAutoModelFor...`) were removed in the 5.x line. Our labs are TensorFlow-first, so we stay on the 4.x branch.
- **`tf-keras`** - TensorFlow 2.16 and later ship **Keras 3** as `tf.keras`. Both Hugging Face's TF models and `tensorflow-model-optimization` (which we need in Labs 3 and 4) are written against the **Keras 2** API. The `tf-keras` package provides Keras 2, and the `TF_USE_LEGACY_KERAS=1` environment variable tells TensorFlow to route `tf.keras` to it.
- **`datasets < 4`** - keeps the loading API stable for the GLUE/SQuAD/IMDB splits we use.

This cell takes 1–2 minutes.

In [ ]:
%pip install -q "transformers>=4.40,<5" "datasets>=2.19,<4" "tf-keras>=2.16" "tensorflow-model-optimization>=0.8.0" "scikit-learn" "pandas"
print('dependencies installed')

### 0.3 Set the legacy-Keras flag *before* importing TensorFlow

`TF_USE_LEGACY_KERAS` is read at import time. If TensorFlow has already been imported in this session, the flag has no effect - restart the runtime and re-run from the top. The next cell checks for exactly that situation and tells you.

In [ ]:
import os, sys

os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # quieten the C++ logger

if "tensorflow" in sys.modules:
    print(
        "TensorFlow was already imported. Runtime > Restart session, then re-run "
        "from the top before continuing."
    )

import tensorflow as tf
import numpy as np
import transformers

print("TensorFlow  :", tf.__version__)
# tf.keras is a lazy loader and does not re-export __version__; ask the
# underlying package instead, and confirm which one tf.keras resolves to.
try:
    import tf_keras as _keras_pkg
except ImportError:
    import keras as _keras_pkg
_keras_impl = tf.keras.Model.__module__  # e.g. "tf_keras.src.models.model"

print("Keras       :", _keras_pkg.__version__, "(needs to start with 2.)")
print("tf.keras is :", _keras_impl, "(needs to contain 'tf_keras')")
assert _keras_pkg.__version__.startswith("2."), (
    "Keras 3 is active. Install tf-keras and set TF_USE_LEGACY_KERAS=1 BEFORE "
    "importing tensorflow, then Runtime > Restart session."
)
assert "tf_keras" in _keras_impl, (
    "TF_USE_LEGACY_KERAS was not picked up. It must be set before the first "
    "tensorflow import. Runtime > Restart session and re-run from the top."
)
print("Transformers:", transformers.__version__)
print("Devices     :", [d.device_type for d in tf.config.list_physical_devices()])

tf.keras.utils.set_random_seed(42)

### 0.4 A persistent artifact root

Colab runtimes are ephemeral. Labs 2–5 need the models and the ledger produced here, so we mount Google Drive and keep everything under one root directory.

If you would rather not mount Drive, set `USE_DRIVE = False` - everything still works, but a runtime disconnect means re-running Lab 1 before continuing.

In [ ]:
USE_DRIVE = True
ROOT = "/content/llm-deploy-labs"

if USE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        ROOT = "/content/drive/MyDrive/llm-deploy-labs"
    except Exception as e:
        print("Drive unavailable, falling back to local storage:", e)

os.environ["LLMDEPLOY_ROOT"] = ROOT
for sub in ("models", "reports", "data"):
    os.makedirs(os.path.join(ROOT, sub), exist_ok=True)
print("Artifact root:", ROOT)

---
## 1. TensorFlow and Keras: the parts that matter at deployment time

This is a refresher, not an introduction - it is deliberately biased toward the constructs that determine whether a model is *servable*, and it skips the ones that only matter during research.

### 1.1 Tensors, dtypes, and device placement

A `tf.Tensor` is an immutable, typed, n-dimensional array with a device placement. Three things about it drive deployment cost:

- **dtype** is memory. A `float32` weight is 4 bytes; `float16` is 2; `int8` is 1. This single fact is the entire basis of Lab 3.
- **shape** is compute. Transformer self-attention is quadratic in sequence length, so padding every request to a fixed 512 tokens when your median input is 20 tokens means you are paying for roughly 600× more attention arithmetic than you need.
- **device** is where the numbers live. Every `.numpy()` call is a synchronous device-to-host copy, which is why we never put one inside a benchmark loop.

In [ ]:
x = tf.constant([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("shape", x.shape, "| dtype", x.dtype, "| device", x.device.split("/")[-1])

# dtype is memory: the same tensor at three precisions
for dt in (tf.float32, tf.float16, tf.int8):
    t = tf.cast(x, dt)
    print(
        f"{dt.name:>8}: {t.dtype.size} bytes/element -> "
        f"{t.dtype.size * int(tf.size(t))} bytes for this tensor"
    )

# Shape is compute: attention FLOPs scale with L^2
for L in (32, 128, 512):
    print(f"seq_len={L:>3} -> attention matrix has {L*L:>7,} entries per head")

### 1.2 `tf.data`: the input pipeline is part of your latency budget

At training time a slow pipeline starves the GPU. At serving time the same logic - tokenization, padding, batching - sits directly on the request path. `tf.data` gives you three primitives worth memorising:

- `.map(fn, num_parallel_calls=AUTOTUNE)` - parallelise per-element work.
- `.cache()` - materialise the result of everything upstream once.
- `.prefetch(AUTOTUNE)` - overlap host-side preparation with device-side compute.

The demo below makes the cost of omitting them visible.

In [ ]:
import time


def slow_square(v):
    # tf.py_function simulates a genuinely expensive host-side step
    # (tokenization, image decode, feature lookup...).
    return tf.py_function(lambda a: a.numpy() ** 2, [v], tf.float32)


base = tf.data.Dataset.range(256).map(lambda i: tf.cast(i, tf.float32))


def consume(ds, label):
    t0 = time.perf_counter()
    for _ in ds:
        pass
    print(f"{label:<34} {time.perf_counter() - t0:6.3f}s")


consume(base.map(slow_square).batch(32), "naive map + batch")
consume(
    base.map(slow_square, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE),
    "parallel map + prefetch",
)

### 1.3 The three Keras model APIs - and which one you can actually export

| API | Written as | Graph known ahead of time? | Matters because |
|---|---|---|---|
| **Sequential** | a list of layers | yes | trivially exportable, prunable, quantizable |
| **Functional** | a graph of tensors | yes | same, plus multi-input/multi-output |
| **Subclassed** | a Python `class` with `call()` | **no** | maximum flexibility, but tooling that needs to *rewrite* the graph often cannot |

That last row is not academic. Hugging Face's `TFBertForSequenceClassification` is a **subclassed** model. In Lab 4 you will find that `tensorflow_model_optimization`'s pruning wrapper refuses to clone it - because there is no static layer graph to clone. We solve that by writing our own masking logic, and by demonstrating the standard API on a Functional model. Knowing *why* it fails is the point.

In [ ]:
# Sequential
seq = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(16,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(2),
    ],
    name="sequential_demo",
)

# Functional - same computation, explicit tensor graph
inp = tf.keras.Input(shape=(16,), name="features")
h = tf.keras.layers.Dense(32, activation="relu")(inp)
out = tf.keras.layers.Dense(2, name="logits")(h)
fnc = tf.keras.Model(inp, out, name="functional_demo")


# Subclassed - flexible, but `fnc.layers`-style graph introspection is limited
class Subclassed(tf.keras.Model):
    def __init__(self):
        super().__init__(name="subclassed_demo")
        self.d1 = tf.keras.layers.Dense(32, activation="relu")
        self.d2 = tf.keras.layers.Dense(2)

    def call(self, x, training=False):
        return self.d2(self.d1(x))


sub = Subclassed()
sub(tf.zeros([1, 16]))  # a subclassed model is only "built" once it is called

for m in (seq, fnc, sub):
    print(
        f"{m.name:<18} params={m.count_params():>6}  "
        f"has_static_graph={getattr(m, 'inputs', None) is not None}"
    )

### 1.4 `GradientTape`: why we train with an explicit loop in these labs

`model.fit()` is excellent when the loss is a simple function of `(y_true, y_pred)`. Every technique on Day 1 breaks that assumption:

- **Distillation** needs the *teacher's* logits inside the loss.
- **Pruning** needs a callback that re-applies weight masks after each optimizer step.
- **Quantization-aware training** needs fake-quant nodes threaded through the forward pass.

So we standardise on one explicit loop, defined once in the lab kit (Section 3) and reused everywhere. Below is the same loop in miniature, so the abstraction is not a black box.

In [ ]:
# Fit y = 3x + 2 with an explicit tape loop.
true_w, true_b = 3.0, 2.0
xs = tf.random.normal([512, 1])
ys = true_w * xs + true_b + tf.random.normal([512, 1], stddev=0.1)

w = tf.Variable(0.0)
b = tf.Variable(0.0)
opt = tf.keras.optimizers.SGD(learning_rate=0.1)


@tf.function
def step(xb, yb):
    with tf.GradientTape() as tape:  # 1. record forward ops
        pred = w * xb + b
        loss = tf.reduce_mean(tf.square(pred - yb))
    grads = tape.gradient(loss, [w, b])  # 2. reverse-mode differentiate
    opt.apply_gradients(zip(grads, [w, b]))  # 3. update
    return loss


ds = tf.data.Dataset.from_tensor_slices((xs, ys)).batch(32).repeat(20)
for i, (xb, yb) in enumerate(ds):
    loss = step(xb, yb)
    if i % 160 == 0:
        print(
            f"step {i:>4}  loss={float(loss):.4f}  w={float(w):.3f}  b={float(b):.3f}"
        )
print(f"\nrecovered w={float(w):.3f} (true {true_w}), b={float(b):.3f} (true {true_b})")

### 1.5 `tf.function`: eager for debugging, graph for serving

Eager execution runs op-by-op through Python. `tf.function` traces your Python once into a dataflow graph, then executes that graph natively - removing the Python interpreter from the hot loop and enabling whole-graph optimizations (constant folding, op fusion, and with `jit_compile=True`, XLA kernel fusion).

Two consequences you must internalise:

- **Retracing is expensive.** A new input shape or dtype triggers a fresh trace. This is exactly why we warm up before benchmarking, and why serving a fixed padded shape can beat serving arbitrary shapes even though it does more arithmetic.
- **Python side effects run only during tracing.** A `print()` inside a `tf.function` fires once, not every call. Use `tf.print` if you need runtime output.

In [ ]:
mlp = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(256,)),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(10),
    ]
)
batch = tf.random.normal([64, 256])

graph_fn = tf.function(lambda x: mlp(x, training=False))
xla_fn = tf.function(lambda x: mlp(x, training=False), jit_compile=True)


def bench(fn, label, warmup=10, runs=200):
    for _ in range(warmup):
        fn(batch)
    t0 = time.perf_counter()
    for _ in range(runs):
        fn(batch)
    print(f"{label:<26} {(time.perf_counter() - t0) / runs * 1000:6.3f} ms/call")


bench(lambda x: mlp(x, training=False), "eager")
bench(graph_fn, "tf.function")
bench(xla_fn, "tf.function + XLA")

### 1.6 `SavedModel` and the serving signature - the Day 2 contract

A `SavedModel` directory is TensorFlow's language-neutral deployment format. It contains the graph, the weights, and - critically - a set of **signatures**: named entry points with fixed input and output specs.

On Day 2, TensorFlow Serving loads exactly this directory and exposes the `serving_default` signature over REST and gRPC. Everything you cannot express in the signature (tokenization, truncation, post-processing) has to live in your Flask layer instead. Deciding where that boundary sits is a deployment design decision, and you make it here, at export time.

Note the `None` in the batch dimension: it lets the served graph accept any batch size without retracing.

In [ ]:
export_dir = os.path.join(ROOT, "models", "_demo_savedmodel")


@tf.function(input_signature=[tf.TensorSpec([None, 256], tf.float32, name="features")])
def serving_fn(features):
    return {"logits": mlp(features, training=False)}


tf.saved_model.save(mlp, export_dir, signatures={"serving_default": serving_fn})

reloaded = tf.saved_model.load(export_dir)
infer = reloaded.signatures["serving_default"]
print(
    "inputs :",
    {k: v.shape.as_list() for k, v in infer.structured_input_signature[1].items()},
)
print("outputs:", {k: v.shape.as_list() for k, v in infer.structured_outputs.items()})
print("\nsame result for batch sizes 1 and 8 without retracing:")
for bs in (1, 8):
    print(" ", bs, "->", infer(features=tf.random.normal([bs, 256]))["logits"].shape)

In [ ]:
# The CLI you will use on Day 2 to sanity-check an artifact before shipping it.
!saved_model_cli show --dir "{export_dir}" --tag_set serve --signature_def serving_default

---
## 2. Case study: scaling a BERT text classifier to a billion requests a day

As per our curriculum, let's go through a success story - our case study. We use Roblox's published account of putting a BERT text classifier into production, because it is one of the few public write-ups that reports before-and-after numbers for the exact three-lever playbook this course teaches.

### 2.1 The situation

Roblox needed real-time text classification for content moderation at platform scale. The research artifact was a fine-tuned BERT model. The production requirement was on the order of **a billion requests per day** at conversational latency - a workload where a GPU fleet sized for peak traffic becomes the dominant line item in the service's budget.

### 2.2 What they changed

Their reported playbook was, in their own framing, about making things smaller. <cite index="4-1">The levers were a smaller model through distillation, smaller inputs through dynamic input shapes, smaller weights through quantization, fewer requests through caching, and tuned thread counts per core.</cite>

Three of those five map directly onto Day 1 of this course:

| Their lever | Mechanism | Our lab |
|---|---|---|
| Smaller model | BERT → DistilBERT (knowledge distillation) | **Lab 2** |
| Smaller weights | dynamic int8 quantization | **Lab 3** |
| Smaller inputs | drop fixed-length padding, batch size 1 | **Labs 1, 5** |
| Fewer requests | cache on token-id key | Day 2 discussion |
| Thread tuning | serving-runtime configuration | Day 2 |

Two details deserve attention because they contradict common intuition:

**Batching made things worse.** <cite index="6-1">Because batching required zero-padding inputs of different lengths to a common length, it turned out to be faster to use a batch size of one, which removed the padding entirely and shortened the inputs considerably.</cite> For a request-response workload with short, highly variable inputs, the wasted arithmetic on padding tokens exceeded the hardware-efficiency gain from batching. This is why our benchmark harness measures batch size 1 and batch size 32 separately - they are different deployment regimes with different answers.

**Quantization was the single biggest win,** and it cost very little accuracy. <cite index="3-1">Combining DistilBERT, dynamic shapes and dynamic quantization produced roughly a 30× improvement in both latency and throughput over the vanilla BERT baseline, with a negative F1 impact of under one percent after quantization.</cite>

### 2.3 The outcome

<cite index="2-1">Their conclusion was that there is a viable scalability story for DistilBERT and BERT on CPU, particularly for real-time text classification, and that serving over a billion requests a day at median latencies under 20 ms was achievable at reasonable cost.</cite> The strategic result is the part to remember: the optimizations did not merely make the GPU deployment cheaper, they **changed the hardware class**. Once the model was small enough and fast enough, CPU serving became viable - and CPU capacity is cheaper, more elastic, and far easier to acquire than accelerator capacity.

### 2.4 What we take from it

1. Optimize in the order **distil → quantize → prune**, measuring after each step. Compounding is not guaranteed and must be verified.
2. Benchmark in the regime you will actually serve in. Batch size 1 with real input-length distributions, not batch size 64 with fixed 512-token padding.
3. The target is not "faster". The target is **a cheaper hardware class at an acceptable quality loss**, with the acceptable loss agreed in advance.

That third point is what the next cell makes concrete.

### 2.5 Exercise: build the cost model

Before optimizing anything, decide what a millisecond is worth. The calculator below turns a latency figure into a monthly infrastructure bill, given a traffic level and an instance price.

Edit the assumptions and answer, in the reflection cell that follows, what latency you would need to hit to justify moving off GPU.

In [ ]:
import pandas as pd

# --- Assumptions you should change to match your own environment -------------
DAILY_REQUESTS = 1_000_000_000  # requests/day
PEAK_FACTOR = 2.5  # peak QPS / average QPS
TARGET_UTIL = 0.60  # never size a fleet at 100% utilisation
HOURS_PER_MONTH = 730

# (instance class, on-demand $/hour, concurrent request streams per instance)
INSTANCE_PRICES = {"gpu": (1.006, 1), "cpu": (0.340, 8)}  # illustrative us-east-1 rates
# -----------------------------------------------------------------------------


def monthly_cost(p50_ms, hardware, price_per_hour=None, streams=None):
    price, conc = INSTANCE_PRICES[hardware]
    price = price_per_hour if price_per_hour is not None else price
    streams = streams if streams is not None else conc

    avg_qps = DAILY_REQUESTS / 86_400
    peak_qps = avg_qps * PEAK_FACTOR
    qps_per_i = (1000.0 / p50_ms) * streams  # requests/sec one instance serves
    instances = peak_qps / (qps_per_i * TARGET_UTIL)
    return {
        "hardware": hardware,
        "p50_ms": p50_ms,
        "qps_per_instance": round(qps_per_i, 1),
        "instances": int(np.ceil(instances)),
        "monthly_usd": round(np.ceil(instances) * price * HOURS_PER_MONTH),
    }


scenarios = [
    ("baseline BERT, GPU", monthly_cost(120, "gpu")),
    ("distilled, GPU", monthly_cost(55, "gpu")),
    ("distilled + quantized, CPU", monthly_cost(20, "cpu")),
    ("+ pruned & tuned, CPU", monthly_cost(12, "cpu")),
]
df = pd.DataFrame([{"scenario": s, **d} for s, d in scenarios])
df["vs_baseline"] = (df["monthly_usd"] / df["monthly_usd"].iloc[0]).round(3)
df

**Reflect (5 minutes, discuss in pairs):**

1. Which single row produces the largest saving - and is it the row that changes the *latency* most, or the row that changes the *hardware class*?
2. Your product owner will accept 1% quality loss but not 3%. Which of distillation, quantization and pruning would you attempt first, and what would you measure to decide whether to keep it?
3. `TARGET_UTIL = 0.60` is doing a lot of work in this model. What happens to the comparison at 0.85, and what operational risk are you taking on to get there?

Keep your answers - Lab 5 asks you to revisit them against the numbers you actually measured.

---
## 3. The lab kit: one measurement harness for all five labs

Optimization work is worthless without trustworthy measurement. The kit below is deliberately small and is written to disk so Labs 2–5 import the identical code.

What it gives you:

- `measure_latency()` - warm-up, repeated runs, and **percentiles** rather than a mean. Tail latency is what your users experience and what your SLO is written against; a mean hides it.
- `train()` - the single `GradientTape` loop from 1.4, parameterised by a `loss_fn` and an optional per-step callback.
- `record()` / `ledger_df()` - a JSON ledger keyed by stage name, so re-running a cell updates a row rather than duplicating it.
- `size_mb()`, `count_params()`, `weight_sparsity()` - the size accounting we compare across labs.

In [ ]:
# Write the shared lab kit to the artifact root so every later notebook
# imports the *same* measurement code.
LABKIT_SRC = r'''
"""
labkit.py - shared utilities for the "Mastering LLM Deployment" hands-on labs.

Everything the labs need in common lives here so that each notebook measures
the same things in the same way:

  * artifact + ledger management (results survive across notebooks via Drive)
  * a model "size on disk" and parameter/sparsity accounting
  * a latency/throughput benchmark harness with warm-up and percentiles
  * a minimal, explicit GradientTape training loop (works for HF TF models,
    plain Keras models, distillation losses and masked/pruned training alike)
"""

import os

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")

import json
import shutil
import time
from pathlib import Path

import numpy as np
import tensorflow as tf

# --------------------------------------------------------------------------
# 1. Artifact root
# --------------------------------------------------------------------------

_ROOT = Path(os.environ.get("LLMDEPLOY_ROOT", "/content/llm-deploy-labs"))


def set_root(path):
    """Point the lab kit at a persistent directory (ideally on Google Drive)."""
    global _ROOT
    _ROOT = Path(path)
    for sub in ("models", "reports", "data"):
        (_ROOT / sub).mkdir(parents=True, exist_ok=True)
    os.environ["LLMDEPLOY_ROOT"] = str(_ROOT)
    return _ROOT


def root():
    return _ROOT


def model_dir(name, clean=False):
    """Return (and create) a directory under <root>/models/<name>."""
    d = _ROOT / "models" / name
    if clean and d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
    return d


# --------------------------------------------------------------------------
# 2. Size and parameter accounting
# --------------------------------------------------------------------------


def size_mb(path):
    """Size of a file or, recursively, of a directory - in MB."""
    p = Path(path)
    if p.is_file():
        return p.stat().st_size / 1e6
    total = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
    return total / 1e6


def count_params(model):
    """Total trainable parameter count."""
    return int(sum(int(np.prod(v.shape)) for v in model.trainable_variables))


def weight_sparsity(model, kinds=("kernel", "weight", "embeddings")):
    """Fraction of zeros across the "real" weight matrices (ignores biases /
    LayerNorm, which are never pruned in practice)."""
    zeros, total = 0, 0
    for v in model.trainable_variables:
        if not any(k in v.name for k in kinds):
            continue
        arr = v.numpy()
        zeros += int((arr == 0).sum())
        total += int(arr.size)
    return zeros / max(total, 1)


# --------------------------------------------------------------------------
# 3. Latency / throughput benchmarking
# --------------------------------------------------------------------------


def measure_latency(predict_fn, inputs, warmup=5, runs=30, batch_size=1):
    """Run predict_fn(inputs) repeatedly and report wall-clock percentiles.

    Warm-up matters: the first calls pay for graph tracing, kernel autotuning
    and (on GPU) cuDNN algorithm selection. Reporting those numbers is the
    single most common benchmarking mistake in deployment work.
    """
    for _ in range(warmup):
        predict_fn(inputs)

    samples = []
    for _ in range(runs):
        t0 = time.perf_counter()
        predict_fn(inputs)
        samples.append((time.perf_counter() - t0) * 1000.0)

    samples = np.array(sorted(samples))
    p50 = float(np.percentile(samples, 50))
    return {
        "mean_ms": round(float(samples.mean()), 2),
        "p50_ms": round(p50, 2),
        "p90_ms": round(float(np.percentile(samples, 90)), 2),
        "p95_ms": round(float(np.percentile(samples, 95)), 2),
        "throughput_rps": round(batch_size / (p50 / 1000.0), 1),
    }


def device_label():
    return "GPU" if tf.config.list_physical_devices("GPU") else "CPU"


# --------------------------------------------------------------------------
# 4. The optimization ledger
# --------------------------------------------------------------------------


def _ledger_file():
    (_ROOT / "reports").mkdir(parents=True, exist_ok=True)
    return _ROOT / "reports" / "ledger.json"


def load_ledger():
    f = _ledger_file()
    if not f.exists():
        return []
    return json.loads(f.read_text())


def record(stage, **fields):
    """Insert or replace a ledger row. Stage names are unique keys, so
    re-running a cell updates the row instead of duplicating it."""
    ledger = [e for e in load_ledger() if e.get("stage") != stage]
    entry = {"stage": stage, "recorded_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    entry.update(fields)
    ledger.append(entry)
    _ledger_file().write_text(json.dumps(ledger, indent=2))
    return entry


def ledger_df(columns=None):
    import pandas as pd

    df = pd.DataFrame(load_ledger())
    if df.empty:
        return df
    preferred = [
        "stage",
        "model",
        "task",
        "dataset",
        "params_m",
        "size_mb",
        "quality",
        "quality_metric",
        "p50_ms",
        "p95_ms",
        "throughput_rps",
        "device",
        "notes",
    ]
    cols = columns or [c for c in preferred if c in df.columns]
    extra = [c for c in df.columns if c not in cols and c != "recorded_at"]
    return df[cols + extra]


# --------------------------------------------------------------------------
# 5. A small, explicit training loop
# --------------------------------------------------------------------------


def train(
    model,
    dataset,
    loss_fn,
    optimizer,
    epochs=1,
    steps_per_epoch=None,
    log_every=50,
    on_step_end=None,
    clip_norm=1.0,
):
    """Generic GradientTape loop.

    loss_fn(model, batch, training) -> scalar loss tensor.
    on_step_end(global_step) -> optional Python callback, used by the pruning
    lab to update sparsity masks between steps.
    """

    @tf.function
    def train_step(batch):
        with tf.GradientTape() as tape:
            loss = loss_fn(model, batch, True)
        grads = tape.gradient(loss, model.trainable_variables)
        pairs = [
            (g, v) for g, v in zip(grads, model.trainable_variables) if g is not None
        ]
        if clip_norm:
            gs, _ = tf.clip_by_global_norm([g for g, _ in pairs], clip_norm)
            pairs = list(zip(gs, [v for _, v in pairs]))
        optimizer.apply_gradients(pairs)
        return loss

    global_step = 0
    history = []
    for epoch in range(epochs):
        running, seen = 0.0, 0
        t0 = time.time()
        for step, batch in enumerate(dataset):
            loss = float(train_step(batch))
            running += loss
            seen += 1
            global_step += 1
            if on_step_end is not None:
                on_step_end(global_step)
            if log_every and global_step % log_every == 0:
                print(
                    f"  epoch {epoch + 1} | step {global_step:>5} | "
                    f"loss {running / seen:.4f}"
                )
                running, seen = 0.0, 0
            if steps_per_epoch and step + 1 >= steps_per_epoch:
                break
        history.append({"epoch": epoch + 1, "seconds": round(time.time() - t0, 1)})
        print(f"  epoch {epoch + 1} finished in {history[-1]['seconds']}s")
    return history


# --------------------------------------------------------------------------
# 6. Evaluation helpers
# --------------------------------------------------------------------------


def evaluate_accuracy(logits_fn, dataset):
    """logits_fn(features) -> array of shape [batch, num_classes]."""
    correct, total = 0, 0
    for features, labels in dataset:
        logits = np.asarray(logits_fn(features))
        preds = logits.argmax(axis=-1)
        labels = np.asarray(labels)
        correct += int((preds == labels).sum())
        total += int(labels.shape[0])
    return correct / max(total, 1)


def hf_logits_fn(model):
    """Wrap a Hugging Face TF model so it returns a plain logits tensor and is
    compiled once into a graph (fair, low-overhead benchmarking)."""

    @tf.function(reduce_retracing=True)
    def fn(features):
        return model(features, training=False).logits

    return fn


def banner(title):
    line = "=" * max(60, len(title) + 4)
    print(f"\n{line}\n  {title}\n{line}")
'''

from pathlib import Path

kit_path = Path(ROOT) / "labkit.py"
kit_path.write_text(LABKIT_SRC)
print("lab kit written to", kit_path)

In [ ]:
import importlib, sys

sys.path.insert(0, ROOT)
import labkit as lk

importlib.reload(lk)

lk.set_root(ROOT)
lk.banner("lab kit ready")
print("root      :", lk.root())
print("device    :", lk.device_label())
print("ledger has", len(lk.load_ledger()), "row(s)")

---
## 4. Build the baseline

### 4.1 The task

We use **SST-2** (Stanford Sentiment Treebank, binary) from the GLUE benchmark: short pieces of text, one binary label. It stands in for the production workload in the case study - high-volume, real-time, short-text classification - while being small enough to fine-tune inside a lab.

Two properties make it the right thread for the whole of Day 1:

- **Short inputs**, so the padding-versus-dynamic-shape trade-off from the case study is visible rather than theoretical.
- **The same label space as IMDB**, which Lab 3 uses for quantization. That lets us quantize this exact model and evaluate it on a *different* text distribution - long reviews instead of short fragments - which is a far more honest test of whether quantization damaged it.

`FAST_MODE` subsamples the training set so the lab fits its time budget. Set it to `False` if you want the full 67k-example run (~10 minutes on a T4) and a stronger teacher.

In [ ]:
from datasets import load_dataset

FAST_MODE = True
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 1
LR = 2e-5
MODEL_NAME = "bert-base-uncased"

raw = load_dataset("nyu-mll/glue", "sst2")
train_raw = raw["train"].shuffle(seed=42)
if FAST_MODE:
    train_raw = train_raw.select(range(20_000))
val_raw = raw["validation"]

print(f"train: {len(train_raw):,}   validation: {len(val_raw):,}")
print("\nexample:", train_raw[0])
print("\ntoken-length distribution of the validation set (whitespace approximation):")
lens = np.array([len(t.split()) for t in val_raw["sentence"]])
print(
    f"  median {np.median(lens):.0f} | p95 {np.percentile(lens, 95):.0f} | max {lens.max()}"
)
print(
    f"  padding every request to {MAX_LEN} tokens wastes roughly "
    f"{(1 - np.median(lens) / MAX_LEN) * 100:.0f}% of the compute at the median."
)

### 4.2 Tokenize and build `tf.data` pipelines

We pad to a fixed `MAX_LEN` here. That is the *wrong* choice for production serving - the case study is explicit about it - but it is the right choice for training throughput, and keeping it fixed makes the Lab 1→5 latency numbers comparable. Lab 5 measures what dynamic shapes buy you.

In [ ]:
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode(texts):
    enc = tokenizer(
        list(texts),
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
        return_tensors="np",
    )
    return {k: np.asarray(v, dtype=np.int32) for k, v in enc.items()}


def make_dataset(texts, labels, batch_size, shuffle=False):
    feats = encode(texts)
    ds = tf.data.Dataset.from_tensor_slices((feats, np.asarray(labels, dtype=np.int32)))
    if shuffle:
        ds = ds.shuffle(4096, seed=42, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(
    train_raw["sentence"], train_raw["label"], BATCH_SIZE, shuffle=True
)
val_ds = make_dataset(val_raw["sentence"], val_raw["label"], 64)

val_features = encode(val_raw["sentence"])  # kept for the latency harness
val_labels = np.asarray(val_raw["label"])
print("feature keys:", list(val_features.keys()))
print("batch spec  :", train_ds.element_spec[0]["input_ids"])

### 4.3 Fine-tune BERT-base

`TFAutoModelForSequenceClassification` loads pretrained encoder weights and attaches a randomly initialised 2-way classification head. The warning about newly initialised weights is expected - that is the head we are about to train.

The learning-rate schedule is a linear decay to zero, the standard recipe for BERT fine-tuning. Gradient clipping at global norm 1.0 is handled inside `lk.train`.

**Expected time on a T4: 6–10 minutes in `FAST_MODE`.**

#### A note on weight formats

The Hub now serves most checkpoints as `model.safetensors`. When you ask for a **TensorFlow** class and the repo has no TF weights, `transformers` converts the PyTorch tensors on the fly - and that converter expects a plain dictionary, while newer `safetensors` hands it a lazy file handle. The result is a confusing `TypeError: 'builtins.safe_open' object is not iterable`.

The loader below sidesteps it: first ask for non-safetensors weights (most repos still ship `tf_model.h5` or `pytorch_model.bin`), and only if that fails, convert the safetensors shards into a real `.bin` locally.

This is worth understanding rather than copying. Framework-conversion friction is one of the recurring hidden costs of a TensorFlow serving stack in a PyTorch-majority ecosystem - a point that comes back on Day 2 when you pick a serving runtime.

In [ ]:
def load_tf_pretrained(cls, name_or_path, **kwargs):
    # Robust TF-model loader.
    #
    # transformers' PyTorch -> TensorFlow converter iterates the state dict
    # directly. Newer safetensors hands it a `safe_open` handle instead of a
    # dict, which is not iterable, so the conversion dies with:
    #     TypeError: 'builtins.safe_open' object is not iterable
    #
    # Two-step strategy:
    #   1. Ask for non-safetensors weights. Most Hub repos still ship
    #      tf_model.h5 or pytorch_model.bin, and either avoids the bug.
    #   2. If only safetensors exist, materialise them into a real .bin on
    #      disk and convert from that, which takes the working code path.
    try:
        return cls.from_pretrained(name_or_path, use_safetensors=False, **kwargs)
    except Exception as first_error:
        print("[loader] direct load failed:", type(first_error).__name__, first_error)
        print("[loader] falling back to manual safetensors conversion")

    import os, glob, shutil, tempfile, torch
    from safetensors.torch import load_file
    from huggingface_hub import snapshot_download

    src = (
        name_or_path
        if os.path.isdir(name_or_path)
        else snapshot_download(
            name_or_path, allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors"]
        )
    )

    shards = sorted(glob.glob(os.path.join(src, "*.safetensors")))
    if not shards:
        raise RuntimeError(f"no safetensors weights found in {src}")

    dst = tempfile.mkdtemp(prefix="tfconv-")
    for fname in os.listdir(src):
        if fname.endswith((".json", ".txt", ".model")) and "index" not in fname:
            shutil.copy(os.path.join(src, fname), dst)

    state = {}
    for shard in shards:
        state.update(load_file(shard))
    torch.save(state, os.path.join(dst, "pytorch_model.bin"))
    print(
        f"[loader] converted {len(shards)} shard(s), {len(state)} tensors -> pytorch_model.bin"
    )

    return cls.from_pretrained(dst, from_pt=True, **kwargs)


teacher = load_tf_pretrained(
    TFAutoModelForSequenceClassification, MODEL_NAME, num_labels=2
)

steps_per_epoch = int(np.ceil(len(train_raw) / BATCH_SIZE))
total_steps = steps_per_epoch * EPOCHS

schedule = tf.keras.optimizers.schedules.PolynomialDecay(
    initial_learning_rate=LR, decay_steps=total_steps, end_learning_rate=0.0
)
optimizer = tf.keras.optimizers.Adam(learning_rate=schedule)

scce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


def classification_loss(model, batch, training):
    features, labels = batch
    logits = model(features, training=training).logits
    return scce(labels, logits)


lk.banner(f"fine-tuning {MODEL_NAME} | {total_steps} steps")
history = lk.train(
    teacher, train_ds, classification_loss, optimizer, epochs=EPOCHS, log_every=100
)

### 4.4 Evaluate quality

Accuracy on the held-out GLUE validation split. A BERT-base fine-tune on the full SST-2 training set lands around 92–93%; the 20k `FAST_MODE` subset typically reaches 90–92%. Whatever you get is your reference point - every later lab is judged against *this* number, not against a published one.

In [ ]:
teacher_logits = lk.hf_logits_fn(teacher)
teacher_acc = lk.evaluate_accuracy(teacher_logits, val_ds)
print(f"baseline validation accuracy: {teacher_acc:.4f}")

### 4.5 Benchmark latency - properly

Three details separate a benchmark you can act on from one you cannot:

1. **Warm up.** The first calls pay for graph tracing and kernel selection. `measure_latency` discards them.
2. **Report percentiles.** Your SLO is written against p95, not the mean.
3. **Separate the regimes.** Batch size 1 is the interactive request-response path. Batch size 32 is the offline/streaming path. The case study found the answer differs between them, so we measure both.

In [ ]:
def slice_features(features, n):
    return {k: tf.constant(v[:n]) for k, v in features.items()}


bs1 = slice_features(val_features, 1)
bs32 = slice_features(val_features, 32)

lat1 = lk.measure_latency(teacher_logits, bs1, warmup=10, runs=50, batch_size=1)
lat32 = lk.measure_latency(teacher_logits, bs32, warmup=10, runs=50, batch_size=32)

print(f"device: {lk.device_label()}")
print("batch=1 :", lat1)
print("batch=32:", lat32)
print(
    f"\nper-request cost at batch 32 is {lat32['p50_ms']/32:.2f} ms "
    f"vs {lat1['p50_ms']:.2f} ms at batch 1"
)

### 4.6 Save the artifact and record the baseline

We save in Hugging Face format (`save_pretrained`), which Labs 3–5 reload directly. The on-disk size is our size baseline: roughly 110 million parameters at 4 bytes each.

**Writing ~440 MB to Drive takes 3–5 minutes.** If you set `USE_DRIVE = False` earlier this is nearly instant, but the artifact will not survive a runtime restart.

In [ ]:
teacher_dir = lk.model_dir("teacher-bert-sst2", clean=True)
teacher.save_pretrained(teacher_dir)
tokenizer.save_pretrained(teacher_dir)

params = lk.count_params(teacher)
disk_mb = lk.size_mb(teacher_dir)
print(f"saved to {teacher_dir}")
print(
    f"parameters: {params/1e6:.1f}M   on disk: {disk_mb:.1f} MB   "
    f"(~{params*4/1e6:.0f} MB expected at fp32)"
)

In [ ]:
lk.record(
    "baseline",
    model="bert-base-uncased (fine-tuned)",
    task="binary sentiment",
    dataset="SST-2",
    params_m=round(params / 1e6, 1),
    size_mb=round(disk_mb, 1),
    quality=round(teacher_acc, 4),
    quality_metric="accuracy",
    p50_ms=lat1["p50_ms"],
    p95_ms=lat1["p95_ms"],
    throughput_rps=lat1["throughput_rps"],
    p50_ms_batch32=lat32["p50_ms"],
    device=lk.device_label(),
    notes="teacher; fixed 128-token padding",
)

lk.ledger_df()

---
## 5. Wrap-up

### What you produced

- `models/teacher-bert-sst2/` - a fine-tuned BERT-base classifier. Labs 3, 4 and 5 all start from it; Lab 2 borrows its training recipe.
- `labkit.py` - the shared measurement harness.
- `reports/ledger.json` - one row so far. By the end of Lab 5 it will have six, and it is what you would put in front of an architecture review.

### The uncomfortable truth about that baseline

Look at the ledger row. Roughly 110M parameters, ~440 MB on disk, and - on a T4, which is a *rented accelerator* - a p50 in the tens of milliseconds at batch size 1. Now re-run the cost model from 2.5 with your own measured number in place of the placeholder `120`.

That is the model we now have to make deployable.

### Checkpoint before moving on

- [ ] `lk.ledger_df()` shows a `baseline` row with a non-null accuracy.
- [ ] `models/teacher-bert-sst2/` exists and is a few hundred MB.
- [ ] You can explain why we measure p95 rather than the mean, and why batch size 1 and 32 are reported separately.
- [ ] You have written down your team's acceptable quality-loss threshold. Lab 5 will hold you to it.

### Next

**Lab 2 - Model Distillation.** We take the largest of the three levers first: training a small student model to reproduce a large teacher's behaviour. Following the syllabus, we do it on **SQuAD** extractive question answering, which forces the distillation loss to handle structured outputs rather than a single label - a more demanding and more transferable version of the technique than sentiment classification alone would give you.